In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

class _Sorter:
    def __init__(self):
        self.edges = []
    def add(self, downstream, upstream):
        self.edges.append((downstream, upstream))

class _GpdStub:
    pd = pd
    def __init__(self, layers):
        self.layers = layers
    def read_file(self, pkg, layer=None):
        # The upstream commit tests construct GeoDataFrame fixtures directly; this
        # stub preserves the same id/toid table shape without requiring geopandas.
        return self.layers[layer].reset_index().copy()

_SIMPLE_FLOWPATHS_PD = pd.DataFrame(
    {"id": ["wb-1", "wb-2"], "toid": ["nex-1", "nex-1"], "geometry": [None, None]}
).set_index("id")
_SIMPLE_NETWORK_PD = pd.DataFrame(
    {"id": ["nex-1", "wb-1", "wb-2"], "toid": [np.nan, "nex-1", "nex-1"], "geometry": [None, None, None]}
).set_index("id")
_SIMPLE_FLOWPATHS_PL = pl.from_pandas(_SIMPLE_FLOWPATHS_PD.reset_index()).rename({"id": "index"})
_SIMPLE_NETWORK_PL = pl.from_pandas(_SIMPLE_NETWORK_PD.reset_index()).rename({"id": "index"})

# --- adjacency_create_matrix_loop ---
FIX_ADJACENCY_CREATE_MATRIX_LOOP_SORTER_BEFORE = _Sorter()
FIX_ADJACENCY_CREATE_MATRIX_LOOP_SORTER_GEN = _Sorter()
FIX_ADJACENCY_CREATE_MATRIX_LOOP_FP_BEFORE = _SIMPLE_FLOWPATHS_PD.copy()
FIX_ADJACENCY_CREATE_MATRIX_LOOP_FP_GEN = _SIMPLE_FLOWPATHS_PL.clone()
FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD = _GpdStub({"flowpaths": _SIMPLE_FLOWPATHS_PD, "network": _SIMPLE_NETWORK_PD})
FIX_ADJACENCY_CREATE_MATRIX_LOOP_NETWORK_BEFORE = _SIMPLE_NETWORK_PD.copy()
FIX_ADJACENCY_CREATE_MATRIX_LOOP_NETWORK_GEN = _SIMPLE_NETWORK_PL.clone()

# --- adjacency_read_file_to_polars ---
FIX_ADJACENCY_READ_FILE_TO_POLARS_ARGS = SimpleNamespace(pkg="in_memory_test_fixture")
FIX_ADJACENCY_READ_FILE_TO_POLARS_GPD = _GpdStub({"flowpaths": _SIMPLE_FLOWPATHS_PD, "network": _SIMPLE_NETWORK_PD})

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_adjacency_create_matrix_loop(sorter, fp, gpd, network):
    for id in fp.index:
        nex = fp.loc[id]["toid"]
        try:
            ds_wb = network.loc[nex]["toid"]
        except KeyError:
            ...
        if isinstance(ds_wb, gpd.pd.Series):
            ds_wb = ds_wb.iloc[0]
        sorter.add(ds_wb, id)
        # mutations:
        network.loc[nex, "toid"] = ds_wb
        network.loc[ds_wb, "toid"] = np.nan
        fp.loc[ds_wb, "toid"] = np.nan
    return None

def before_adjacency_read_file_to_polars(args, gpd):
    fp = gpd.read_file(args.pkg, layer="flowpaths").set_index("id")
    network = gpd.read_file(args.pkg, layer="network").set_index("id")
    return network

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_adjacency_create_matrix_loop(sorter, fp, gpd, network):
    for id in fp.get_column("index").to_list():
        nex = fp.filter(pl.col("index") == id).select("toid").to_series()[0]
        try:
            ds_wb_df = network.filter(pl.col("index") == nex).select("toid")
            if ds_wb_df.height == 0:
                raise KeyError
            ds_wb = ds_wb_df.to_series()[0]
        except KeyError:
            ...
        if isinstance(ds_wb, list):
            ds_wb = ds_wb[0]
        sorter.add(ds_wb, id)
        # mutations:
        network = network.with_columns(
            pl.when(pl.col("index") == nex).then(pl.lit(ds_wb)).otherwise(pl.col("toid")).alias("toid")
        )
        network = network.with_columns(
            pl.when(pl.col("index") == ds_wb).then(pl.lit(None)).otherwise(pl.col("toid")).alias("toid")
        )
        fp = fp.with_columns(
            pl.when(pl.col("index") == ds_wb).then(pl.lit(None)).otherwise(pl.col("toid")).alias("toid")
        )
    return None

def gen_adjacency_read_file_to_polars(args, gpd):
    fp = gpd.read_file(args.pkg, layer="flowpaths").sort_values("id")
    network = gpd.read_file(args.pkg, layer="network").sort_values("id")
    return network

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: adjacency_create_matrix_loop ===

# L1 smoke – generated
try:
    _r = gen_adjacency_create_matrix_loop(FIX_ADJACENCY_CREATE_MATRIX_LOOP_SORTER_GEN, FIX_ADJACENCY_CREATE_MATRIX_LOOP_FP_GEN.clone(), FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD, FIX_ADJACENCY_CREATE_MATRIX_LOOP_NETWORK_GEN.clone())
    print("✅ L1 smoke gen_adjacency_create_matrix_loop: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_adjacency_create_matrix_loop: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_adjacency_create_matrix_loop(FIX_ADJACENCY_CREATE_MATRIX_LOOP_SORTER_BEFORE, FIX_ADJACENCY_CREATE_MATRIX_LOOP_FP_BEFORE.copy(), FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD, FIX_ADJACENCY_CREATE_MATRIX_LOOP_NETWORK_BEFORE.copy())
    print("✅ L1 smoke before_adjacency_create_matrix_loop: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_adjacency_create_matrix_loop: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – this snippet mutates sorter/fp/network, so compare side effects.
try:
    _before_sorter = _Sorter()
    _gen_sorter = _Sorter()
    _before_fp = FIX_ADJACENCY_CREATE_MATRIX_LOOP_FP_BEFORE.copy()
    _before_network = FIX_ADJACENCY_CREATE_MATRIX_LOOP_NETWORK_BEFORE.copy()
    _gen_fp = FIX_ADJACENCY_CREATE_MATRIX_LOOP_FP_GEN.clone()
    _gen_network = FIX_ADJACENCY_CREATE_MATRIX_LOOP_NETWORK_GEN.clone()
    before_adjacency_create_matrix_loop(_before_sorter, _before_fp, FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD, _before_network)
    gen_adjacency_create_matrix_loop(_gen_sorter, _gen_fp, FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD, _gen_network)

    def _norm_edges(edges):
        return [(None if pd.isna(downstream) else downstream, upstream) for downstream, upstream in edges]

    _edges_match = _norm_edges(_before_sorter.edges) == _norm_edges(_gen_sorter.edges)
    if _edges_match:
        print("✅ L2 equivalence adjacency_create_matrix_loop sorter side effect: MATCH")
    else:
        print(f"❌ L2 equivalence adjacency_create_matrix_loop sorter side effect: MISMATCH — before={_before_sorter.edges}, gen={_gen_sorter.edges}")

    _before_fp_cmp = _before_fp.reset_index().rename(columns={"id": "index"})
    _before_network_cmp = _before_network.reset_index().rename(columns={"id": "index"})
    compare(_before_fp_cmp, _gen_fp, "adjacency_create_matrix_loop fp side effect", check_row_order=True)
    compare(_before_network_cmp, _gen_network, "adjacency_create_matrix_loop network side effect", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence adjacency_create_matrix_loop: setup error — {type(_e).__name__}: {_e}")

# L3 edge - compare empty-input side effects, not only the None return.
try:
    _bs, _gs = _Sorter(), _Sorter()
    _bfp = pd.DataFrame({"id": pd.Series(dtype="object"), "toid": pd.Series(dtype="object"), "geometry": pd.Series(dtype="object")}).set_index("id")
    _bn = _bfp.copy()
    _gfp = pl.DataFrame(schema={"index": pl.String, "toid": pl.String, "geometry": pl.Null})
    _gn = _gfp.clone()
    _rb = before_adjacency_create_matrix_loop(_bs, _bfp, FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD, _bn)
    _rg = gen_adjacency_create_matrix_loop(_gs, _gfp, FIX_ADJACENCY_CREATE_MATRIX_LOOP_GPD, _gn)
    compare(_bfp.reset_index().rename(columns={"id": "index"}), _gfp, "L3 edge adjacency_create_matrix_loop fp", check_row_order=True)
    compare(_bn.reset_index().rename(columns={"id": "index"}), _gn, "L3 edge adjacency_create_matrix_loop network", check_row_order=True)
    if _rb is None and _rg is None and _bs.edges == _gs.edges == []:
        print("✅ L3 edge adjacency_create_matrix_loop sorter: MATCH")
    else:
        print(f"❌ L3 edge adjacency_create_matrix_loop sorter: MISMATCH — before={_bs.edges}, gen={_gs.edges}")
except Exception as _e:
    print(f"❌ L3 edge adjacency_create_matrix_loop empty input: {type(_e).__name__}: {_e}")
